# COMPASS preprocessing

Stage 0-3 of the COMPASS survival pipeline: schema audit (profile_data only),
cohort compile, longitudinal lab preprocessing, prediction-input build, and
cohort diagnostics. Univariate/multivariate modeling live in
`02_univariate.ipynb` / `03_multivariate.ipynb` and only read the
`prediction_inputs_<arm>/` files this notebook writes. All stages use the
merged `profile_data` parquets.

In [ ]:
ARMS = ["adt"]
ENDPOINTS = ("platinum", "nepc", "avpc_nepc")
# Stage 3 writes one independent tree per endpoint. Each build applies only its
# own time-validity gate: t_platinum never filters NEPC and t_nepc never filters
# platinum. Stages 0-2 remain shared and run once.

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

RUNS = cp.make_endpoint_runs(ARMS, endpoints=ENDPOINTS)

## Stage 0 -- schema audit

Fails fast if a required column is absent or all-null in the merged sources.

In [ ]:
cp.audit_schema()

## Stage 1 -- compile COMPASS cohort data

Endpoint-independent: writes one survival cohort carrying **both** endpoints'
columns (`PLATINUM`/`TT_PLATINUM` and `NEPC`/`TT_NEPC`). The NEPC columns come
from the strict LLM diagnosis labels
(`LLM_annotations/LLM_nepc_diagnosis/nepc_dx_labels.parquet`); if that file is
not mounted the stage still succeeds and simply emits no NEPC columns, leaving
the platinum pipeline unaffected.

Read the printed summary before spending modelling effort: it reports the NEPC
positive count, the `date_source` / `date_precision` / `label_source`
breakdowns, and how many diagnoses are **prevalent** (at or before the anchor)
and will therefore be excluded by the incident-endpoint filter.

In [ ]:
cp.compile_cohort(arms=ARMS)

## Stage 2 -- preprocess raw labs (per arm anchor)

Expensive: full raw lab standardization. The Parquet cache
(`consolidated_longitudinal_data_<arm>.parquet`) makes reruns cheap, but the
first pass may be slow.

In [ ]:
for run in RUNS:
    if run["endpoint"] != "platinum":
        continue  # endpoint-independent stage; run once per arm
    cp.preprocess_labs(run)

## Stage 3 -- build prediction inputs + cohort diagnostics

Set `REBUILD_PREDICTION_INPUTS = False` to skip rebuilding. This cell builds both endpoint trees. Platinum requires only valid `t_platinum`; NEPC requires only valid incident `t_nepc`. Shared death/follow-up checks still apply to both.

In [ ]:
REBUILD_PREDICTION_INPUTS = True

for run in RUNS:
    if REBUILD_PREDICTION_INPUTS:
        cp.build_prediction_inputs(run)
    else:
        print(f"[skip] prediction-input rebuild disabled for {run['label']}")
    cp.cohort_diagnostics(run)

## Stage 3b -- sequencing, Gleason, and PRS inputs

Builds `prediction_inputs_<arm>/somatic_gleason/` from the sample-level
somatic matrix published by `PROFILE_data_processing` and the Gleason timeline
published by `LLM_clinical_annotations`. It creates two cohorts: the sequencing
sample closest to ADT start with follow-up from specimen collection, and the
Gleason score closest to ADT start with follow-up from the score date. PRSs use
ADT start itself as the prediction origin.

In [ ]:
for run in RUNS:
    cp.build_somatic_gleason_inputs(run)